In [1]:
from sklearn.model_selection import train_test_split
from neuroCombat import neuroCombat, neuroCombatFromTraining
import pandas as pd
import numpy as np
import copy
import numpy.linalg as la


In [2]:
#import dataset
#SOURCE
expr_file1= r"C:\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Data\cptacvssmc\cptac_labels.csv"
expr1 = pd.read_csv(expr_file1, encoding='utf-8', engine='python')

#TARGET
expr_file2= r"C:\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Data\cptacvssmc\smc_labels.csv"
expr2 = pd.read_csv(expr_file2, encoding='utf-8', engine='python')


In [6]:
expr1.set_index('Hugo_Symbol',inplace=True)
expr2.set_index('Hugo_Symbol',inplace=True)
print(expr2.index[0])


7SK


In [ ]:
print(expr1.index[0])

In [7]:
genes1 = (expr1.index).tolist()
genes2 = (expr2.index).tolist()

In [8]:
common_genes = sorted(list(set(genes1).intersection(set(genes2))))
if len(common_genes) == 0:
        raise ValueError("No common genes found between source and target datasets.")

In [6]:
X_S=expr1.loc[common_genes].sort_index()
X_T=expr2.loc[common_genes].sort_index()

In [ ]:
X_T

In [7]:
X_Target=X_T.T
X_Source=X_S.T

In [ ]:
X_Source

In [8]:
target_train, target_test = train_test_split(X_Target, test_size=0.2, random_state=42)

In [9]:
# Combine source + target_train
combined = pd.concat([X_Source, target_train])
batch_labels = [1] * len(X_Source) + [2] * len(target_train)


In [ ]:
target_train.shape

In [10]:
covar= pd.DataFrame({'batch': batch_labels})
one_hot = pd.get_dummies(covar, columns=['batch'],dtype=float)
one_hot = one_hot.iloc[:, :2].to_numpy()

In [11]:
combat_result = neuroCombat(
    dat=combined.T,  # genes as rows
    batch_col="batch",
    covars=covar
)
aligned_train_data = combat_result['data'].T  # ComBat corrected source + target_trai

[neuroCombat] Creating design matrix
[neuroCombat] Standardizing data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Final adjustment of data


In [25]:
estimates=combat_result['estimates']
estimates

{'gamma_hat': array([[ 0.87614429,  0.02057762,  0.05557902, ...,  0.01775787,
          0.06010337,  0.02955308],
        [-0.77658244, -0.01823925, -0.04926322, ..., -0.01573993,
         -0.05327345, -0.02619478]]),
 'delta_hat': [array([1.47737471, 0.97422434, 1.10023757, ..., 0.98506648, 0.8534374 ,
         1.01950346]),
  array([0.5925537 , 1.03809142, 0.92650719, ..., 1.02849075, 1.1450478 ,
         0.99799694])],
 'gamma_bar': array([ 0.10679517, -0.09465935]),
 't2': array([0.12791911, 0.10049844]),
 'a_prior': [19.308271046331907, 26.21507039448],
 'b_prior': [18.00413226666027, 25.970945694181765],
 'batches': array([1, 2], dtype=object),
 'var.pooled': array([[0.69438734],
        [1.02324408],
        [0.9496231 ],
        ...,
        [1.03935742],
        [0.99300568],
        [1.02582854]]),
 'stand.mean': array([[-0.70148516, -0.70148516, -0.70148516, ..., -0.70148516,
         -0.70148516, -0.70148516],
        [-0.05523408, -0.05523408, -0.05523408, ..., -0.0552340

In [ ]:
estimates['gamma.star']

### Doing it on the Test Batch

In [17]:
batch_labels_test= [1] * len(target_test)
covar_test = pd.DataFrame({'batch': batch_labels_test})
one_hot_test = pd.get_dummies(covar_test, columns=['batch'], dtype=float)
one_hot_test = one_hot_test.iloc[:, :1].to_numpy()

In [16]:
test_batch=np.zeros((len(target_test), 2))
test_batch[:, 0] = one_hot_test[:, 0]
test_batch

NameError: name 'one_hot_test' is not defined

In [18]:
one_hot_test

array([[1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.]])

In [14]:
batch_info = [list(range(34))]
batch_info

[[0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33]]

In [21]:
if isinstance(target_test, pd.DataFrame):
        print("target_test is a DataFrame")
        dat = np.array(target_test, dtype='float32')

target_test is a DataFrame


In [ ]:
dat.shape

In [ ]:
def standardize_across_features(X, design,batch_info):
    n_batch = 1                             #info_dict['n_batch']
    n_sample =34                            #info_dict['n_sample']
    sample_per_batch = 34                   #info_dict['sample_per_batch']
    batch_info = batch_info                 #info_dict['batch_info']
    ref_level = None                        #info_dict['ref_level']

    def get_beta_with_nan(yy, mod):
        wh = np.isfinite(yy)
        mod = mod[wh,:]
        yy = yy[wh]
        B1=np.dot(mod.T, mod)
        #B2=la.inv(B1) possible value: .00757576
        B3=np.dot(B1, mod.T)
        B4=np.dot(B3, yy.T)
        #B = np.dot(np.dot(la.inv(np.dot(mod.T, mod)), mod.T), yy.T)
        return B4

    betas = []
    for i in range(X.shape[0]):
        #print(X[i,:])
        betas.append(get_beta_with_nan(X[i,:], design))
    B_hat = np.vstack(betas).T
    #print(betas)
    
    #B_hat = np.dot(np.dot(la.inv(np.dot(design.T, design)), design.T), X.T)
    #print( B_hat[:n_batch,:])
    if ref_level is not None:
        grand_mean = np.transpose(B_hat[ref_level,:])
    else:
        grand_mean = np.dot((sample_per_batch/ float(n_sample)), B_hat[:n_batch,:])
        grand_mean=grand_mean.flatten()
    stand_mean = np.dot(grand_mean.T.reshape((len(grand_mean), 1)), np.ones((1, n_sample)))
    #var_pooled = np.dot(((X - np.dot(design, B_hat).T)**2), np.ones((n_sample, 1)) / float(n_sample))
    print(stand_mean.shape)

    if ref_level is not None:
        X_ref = X[:,batch_info[ref_level]]
        design_ref = design[batch_info[ref_level],:]
        n_sample_ref = sample_per_batch[ref_level]
        var_pooled = np.dot(((X_ref - np.dot(design_ref, B_hat).T)**2), np.ones((n_sample_ref, 1)) / float(n_sample_ref))
    else:
        var_pooled = np.dot(((X - np.dot(design, B_hat).T)**2), np.ones((n_sample, 1)) / float(n_sample))

    var_pooled[var_pooled==0] = np.median(var_pooled!=0)
    
    mod_mean = 0
    if design is not None:
        tmp = copy.deepcopy(design)
        tmp[:,range(0,n_batch)] = 0
        mod_mean = np.transpose(np.dot(tmp, B_hat))
    ######### Continue here. 


    #tmp = np.array(design.copy())
    #tmp[:,:n_batch] = 0
    #stand_mean  += np.dot(tmp, B_hat).T

    s_data = ((X- stand_mean - mod_mean) / np.dot(np.sqrt(var_pooled), np.ones((1, n_sample))))

    return s_data, stand_mean, var_pooled, mod_mean

In [ ]:
s_data_t, s_mean_t, v_pool_t, mod_mean_t = standardize_across_features(dat.T,test_batch,batch_info)


In [ ]:
def adjust_data_final(s_data, design, gamma_star, delta_star, stand_mean, mod_mean, var_pooled,batch_info, dat):
    sample_per_batch = 34
    n_batch = 1
    n_sample = 34
    batch_info = batch_info
    ref_level = None

    batch_design = design[:,:n_batch]

    bayesdata = s_data
    gamma_star = np.array(gamma_star)
    delta_star = np.array(delta_star)

    for j, batch_idxs in enumerate(batch_info):
        dsq = np.sqrt(delta_star[j,:])
        dsq = dsq.reshape((len(dsq), 1))
        denom = np.dot(dsq, np.ones((1, sample_per_batch[j])))
        numer = np.array(bayesdata[:,batch_idxs] - np.dot(batch_design[batch_idxs,:], gamma_star).T)

        bayesdata[:,batch_idxs] = numer / denom

    vpsq = np.sqrt(var_pooled).reshape((len(var_pooled), 1))
    bayesdata = bayesdata * np.dot(vpsq, np.ones((1, n_sample))) + stand_mean + mod_mean

    if ref_level is not None:
        bayesdata[:, batch_info[ref_level]] = dat[:,batch_info[ref_level]]

    return bayesdata

In [ ]:
bayes_data_t = adjust_data_final(s_data_t, test_batch, estimates['gamma.star'], estimates['delta.star'], 
                                    s_mean_t, mod_mean_t, v_pool_t,batch_info,dat.T)

In [22]:
test_modified= neuroCombatFromTraining(dat.T, batch_labels_test, estimates)

[neuroCombatFromTraining] In development ...



In [23]:
test_modified

{'data': array([[ 0.33205797, -2.05869512, -2.05869512, ...,  0.67523975,
         -2.05869512, -2.05869512],
        [ 0.28419081,  0.85123598,  0.70736796, ..., -0.26508666,
          0.21220601,  2.30017116],
        [ 0.1477456 , -0.26820225, -1.69335411, ...,  1.08961127,
         -0.07597292, -2.5423993 ],
        ...,
        [ 0.95354631,  0.76397594, -0.34331722, ..., -0.06416852,
          0.63850561,  0.93120225],
        [ 0.55293503,  0.11143066, -0.47034387, ...,  0.11889753,
         -0.12697529,  0.16284529],
        [ 0.21552794, -0.23201621, -1.58233218, ...,  0.48954271,
         -0.20297822, -0.90767315]]),
 'estimates': {'gamma_hat': array([[ 0.87614429,  0.02057762,  0.05557902, ...,  0.01775787,
           0.06010337,  0.02955308],
         [-0.77658244, -0.01823925, -0.04926322, ..., -0.01573993,
          -0.05327345, -0.02619478]]),
  'delta_hat': [array([1.47737471, 0.97422434, 1.10023757, ..., 0.98506648, 0.8534374 ,
          1.01950346]),
   array([0.59255